# Try your detoxification model
Enter a sentence and get a neutral rewrite using your saved T5-small model. **This notebook does not train anything.**

Run this locally in VS Code using the project’s `.venv` Python kernel. Keep the notebook and your model folder or ZIP in the project directory. No NVIDIA GPU is needed.

## 1. Set up your local environment
Dependencies are managed in `pyproject.toml`. From a terminal in the project directory, run:

```bash
uv sync
```

Then select **Select Kernel → Python Environments → .venv/bin/python** in VS Code. If it is not listed, use **Enter interpreter path** and choose that file. The project uses Python 3.12 by default. Run the cell below to confirm the selected interpreter. Installation needs internet; inference uses local model files.

In [1]:
import sys
print("Python:", sys.executable)
print("Version:", sys.version.split()[0])

Python: /Users/ayman/projects/second-thought/.venv/bin/python
Version: 3.12.13


## 2. Choose your saved model
Set `MODEL_DIR` to the saved model folder you want to use, for example `Path("./epoch5")`. The default below uses your original three-epoch model; its folder is named `t5-small-paradetox` even though you renamed its ZIP to `epoch3.zip`.

If the chosen folder does not exist but its ZIP is beside it, this cell extracts the ZIP. An existing folder is never overwritten. If the working directory differs from the project folder, use an absolute path.

In [2]:
from pathlib import Path
import zipfile

MODEL_DIR = Path("./t5-small-paradetox")  # Original three-epoch export.
PREFIX = "detoxify: "  # Must match the training prefix.
MAX_INPUT_TOKENS = 128
MAX_NEW_TOKENS = 128
NUM_BEAMS = 4

MODEL_DIR = MODEL_DIR.expanduser().resolve()
archive_path = MODEL_DIR.with_suffix(".zip")
if not MODEL_DIR.exists():
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Cannot find {MODEL_DIR} or {archive_path}. "
            "Choose the path to your extracted local model folder."
        )
    with zipfile.ZipFile(archive_path) as archive:
        # Check paths before extracting the model folder.
        for member in archive.infolist():
            destination = (MODEL_DIR.parent / member.filename).resolve()
            if not destination.is_relative_to(MODEL_DIR):
                raise ValueError(f"Unexpected ZIP member: {member.filename}")
        archive.extractall(MODEL_DIR.parent)
    print("Extracted:", archive_path.name)

required = ["config.json", "model.safetensors", "tokenizer_config.json", "tokenizer.json"]
missing = [name for name in required if not (MODEL_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete model folder {MODEL_DIR}: missing {missing}")
print("Model folder:", MODEL_DIR)

Model folder: /Users/ayman/projects/second-thought/t5-small-paradetox


## 3. Load the model once
Inference uses your local CPU. Loading happens once; subsequent sentences reuse the model. No NVIDIA GPU, Hugging Face login, or model download is needed.

In [3]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = torch.device("cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR, local_files_only=True)
model.to(device)
model.eval()
print(f"Ready! Using {device}.")

/Users/ayman/projects/second-thought/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready! Using cpu.


## 4. Define the rewriting function
Use the same `detoxify: ` prefix as training. Beam search runs without random sampling. Blank input returns an empty string; inputs over 128 tokens are rejected so text is not silently dropped.

In [4]:
@torch.inference_mode()
def detoxify(text):
    """Rewrite one sentence with the saved model."""
    if not isinstance(text, str):
        raise TypeError("Please provide a string.")
    text = text.strip()
    if not text:
        return ""

    inputs = tokenizer(PREFIX + text, return_tensors="pt", truncation=False)
    if inputs["input_ids"].shape[1] > MAX_INPUT_TOKENS:
        raise ValueError("That text is too long. Please try a shorter sentence.")
    inputs = inputs.to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        do_sample=False,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

## 5. Quick example
Run this to check that loading and generation work. A successful run verifies inference, not the quality of every rewrite.

In [5]:
example = "This is a stupid idea and you never listen."
print("Original:   ", example)
print("Detoxified: ", detoxify(example))

Original:    This is a stupid idea and you never listen.
Detoxified:  This is a bad idea and you never listen.


## 6. Type your own sentence
Run the cell below. In VS Code, type into the input box that appears near the top of the window and press **Enter**. Rerun **only this cell** for another sentence.

Alternatively, call `detoxify("your sentence here")` in a new cell.

Review each result: the model may retain toxic wording or change meaning, and it may correctly leave an already-neutral sentence unchanged.

In [14]:
sentence = "I'd say its a punk though" # input("Enter a sentence to detoxify: ")
if not sentence.strip():
    print("No sentence entered. Rerun this cell to try again.")
else:
    try:
        print("Original:   ", sentence)
        print("Detoxified: ", detoxify(sentence))
    except ValueError as error:
        print(error)

Original:    I'd say its a punk though
Detoxified:  I'd say it's bad though
